# 2.02 - Places of Worship Dataset
## **Source_Dataset**
**places_of_worship.geojson**

The dataset contains locations and attributes for various places of worship, including churches, mosques, temples, and other religious buildings. This dataset tracks the geographic and religious affiliations of these places, making it useful for spatial analysis and demographic studies.


*   Source: https://hub.arcgis.com/datasets/openstreetmap::openstreetmap-places-of-worship-for-north-america/about
*   MimeType: /application/geo+json


Columns:

* Name: Name of the place of worship.
* Latitude: Latitude of the place of worship (geospatial data).
* Longitude: Longitude of the place of worship (geospatial data).
* Denomination: The denomination of the place of worship (e.g., Catholic, Protestant, Sunni, Shia, etc.).
* Religion: The religion associated with the place of worship (e.g., Christianity, Islam, Hinduism).
* Address: The physical address of the place of worship.
* City: The city where the place of worship is located.
* State: The state where the place of worship is located.
* Country: The country where the place of worship is located.

## Output Files
**places_of_worship_with_features.geojson**

This output file contains the Places of Worship dataset augmented with additional features derived from the spatial relationship with haunted places. The new features capture proximity and religion intersection, providing insights into the spatial dynamics between places of worship and nearby haunted locations.

Columns:

* Name: Name of the place of worship.
* Latitude: Latitude of the place of worship.
* Longitude: Longitude of the place of worship.
* Denomination: The denomination of the place of worship.
* Religion: The religion of the place of worship.
* Distance_to_Nearest_Worship: Distance from each haunted place to the nearest place of worship (calculated using geospatial methods like * Haversine distance).
* Haunted_Place_Proximity: Boolean flag indicating whether any haunted places are within 5 miles of the place of worship.

# Features Added to Places_of_Worship.geojson
### **Proximity Features**

* Distance_to_Nearest_Worship: The calculated distance from each haunted place to the nearest place of worship using geospatial distance methods.
* Haunted_Place_Proximity: Boolean flag indicating whether there is at least one haunted place within 5 miles of the place of worship. This is determined by checking the distance to each haunted place and flagging those within the 5-mile radius (converted to meters).

Determining Proximity: The proximity between places of worship and haunted places is calculated using the Haversine distance or a similar geospatial method. A specified radius (e.g., 5 miles) is used to flag whether a haunted place lies within a specified distance from a place of worship.

### **Religious Features**

* Religion_Intersection: The religion of the place of worship closest to each haunted place. This feature is derived by identifying the nearest place of worship based on geospatial distance and mapping the corresponding religion.

# Get the Dataset

In [12]:
import pandas as pd
import geopandas as gpd

## Outfile CSV ##
outfile = "../data/processed/haunted_places_features_added.tab"

## Load Places of Worship Dataset ##
# For Places of Worship, load the GeoJSON file into a GeoDataFrame using geopandas
places_of_worship_df = gpd.read_file("../data/joined_datasets/Places_of_Worship.geojson")

## Load Haunted Places Dataset ##
haunted_places_df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")

## Feature Names ##
feature_names = ["Haunted_Place_Proximity", "Distance_to_Nearest_Worship", "Religion_Intersection"]

# Print out first few rows of the datasets to verify correct loading
print("Places of Worship Dataset:")
print(places_of_worship_df.head())

print("\nHaunted Places Dataset:")
print(haunted_places_df.head())

Places of Worship Dataset:
   objectid     osm_id2 addr_housename addr_housenumber addr_street addr_city  \
0        70  4636123056           None             None        None      None   
1        71  4636123049           None             None        None      None   
2        73  4636123048           None             None        None      None   
3        91  4636123029           None             None        None      None   
4      1160  6330234462           None             None        None      None   

  addr_state addr_postcode addr_province addr_country addr_unit  \
0       None          None          None         None      None   
1       None          None          None         None      None   
2       None          None          None         None      None   
3       None          None          None         None      None   
4       None          None          None         None      None   

            amenity denomination                               name  \
0  place_of_

# Creating Features

In [15]:
from shapely.geometry import Point

# Reproject both GeoDataFrames to a projected CRS (EPSG:3395 - Mercator projection)
places_of_worship_df = places_of_worship_df.to_crs(epsg=3395)  # Reproject to a global Mercator projection
haunted_places_gdf = gpd.GeoDataFrame(
    haunted_places_df,
    geometry=gpd.GeoSeries.from_xy(haunted_places_df['Longitude'], haunted_places_df['Latitude'])
)
haunted_places_gdf.set_crs("EPSG:4326", allow_override=True, inplace=True)  # Set initial CRS for haunted places
haunted_places_gdf = haunted_places_gdf.to_crs(epsg=3395)  # Reproject to Mercator

# Calculate the distance from each haunted place to the nearest place of worship
haunted_places_gdf['Distance_to_Nearest_Worship'] = haunted_places_gdf.geometry.apply(
    lambda x: places_of_worship_df.geometry.distance(x).min() if not places_of_worship_df.geometry.distance(x).isnull().all() else float('nan')
)

# Adding a boolean flag for proximity (e.g., within 5 miles)
# Convert miles to meters (1 mile = 1609.34 meters)
haunted_places_gdf['Haunted_Place_Proximity'] = haunted_places_gdf['Distance_to_Nearest_Worship'] < (5 * 1609.34)  # 5 miles in meters

# Round the Distance_to_Nearest_Worship to 2 decimal places
haunted_places_gdf['Distance_to_Nearest_Worship'] = haunted_places_gdf['Distance_to_Nearest_Worship'].round(2)

# Adding religion intersection feature
haunted_places_gdf['Religion_Intersection'] = haunted_places_gdf.apply(
    lambda row: places_of_worship_df.loc[places_of_worship_df.geometry.distance(row.geometry).idxmin(skipna=True)]['religion']
    if pd.notna(row['Distance_to_Nearest_Worship']) else 'No Religion', axis=1
)

# Preview the modified haunted places GeoDataFrame with the new features
print("\nModified Haunted Places Dataset with Features:")
print(haunted_places_gdf[['Distance_to_Nearest_Worship', 'Haunted_Place_Proximity', 'Religion_Intersection']].head())


Modified Haunted Places Dataset with Features:
   Distance_to_Nearest_Worship  Haunted_Place_Proximity Religion_Intersection
0                      9069.88                    False             christian
1                      4420.08                     True             christian
2                       552.08                     True             christian
3                       255.71                     True             christian
4                      1686.25                     True             christian


# Adding Features to Haunted Places Dataset

In [18]:
# Add the proximity data to the Haunted Places dataset
haunted_places_df['Haunted_Place_Proximity'] = haunted_places_gdf['Haunted_Place_Proximity']
haunted_places_df['Distance_to_Nearest_Worship'] = haunted_places_gdf['Distance_to_Nearest_Worship']
haunted_places_df['Religion_Intersection'] = haunted_places_gdf['Religion_Intersection']

# Preview the modified haunted places GeoDataFrame with the new features
print("\nModified Haunted Places Dataset with Features:")
print(haunted_places_df[['Distance_to_Nearest_Worship', 'Haunted_Place_Proximity', 'Religion_Intersection']].head())

## Save CSV ##
print("Saving to CSV...")

# Read Feature added dataframe
out_df = pd.read_csv(f"{outfile}", sep = "\t")

# Check if feature exists
for feature in feature_names:

    # If it exists, update values
    if feature in out_df.columns:
        out_df[feature].update(haunted_places_df[feature].values)

    # If not add entire column
    else:
        out_df[feature] = haunted_places_df.loc[:,feature]

out_df.to_csv(f"{outfile}", sep = "\t", index = False)

print(f"CSV Saved to {outfile}")


Modified Haunted Places Dataset with Features:
   Distance_to_Nearest_Worship  Haunted_Place_Proximity Religion_Intersection
0                      9069.88                    False             christian
1                      4420.08                     True             christian
2                       552.08                     True             christian
3                       255.71                     True             christian
4                      1686.25                     True             christian
Saving to CSV...
CSV Saved to ../data/processed/haunted_places_features_added.tab
